# The Laplace Transform, A Deeper Look

**Learning Goals**
- Understand the formal definition and motivation for the Laplace transform
- Know the Laplace transforms of the most common signals
- Master the key properties: linearity, differentiation, integration, shifting
- Compute inverse Laplace transforms using partial fraction expansion
- Solve ordinary differential equations (ODEs) using the Laplace transform
- Apply the initial value theorem (IVT) and final value theorem (FVT)

## Relevant lecture video

In [1]:
from IPython.display import HTML

HTML('<iframe width="560" height="315" src="https://echo360.org/media/5570fd92-13fb-4103-b67f-2024a0d86b7d/public?autoplay=false&automute=false&currentMediaId=e0f80dd7-190d-422c-b2d3-8ac6caf6d391" frameborder="0" allowfullscreen></iframe>')

/home/matvei/JupyterBasedControlEngineeringTextbook/venv/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


---

## Why another look at the Laplace transform?

In the time-domain notebook we saw that the Laplace transform turns differential equations into algebraic equations. That alone makes it indispensable for control engineering. But there is much more:

- It reveals the **frequency content** of signals and systems.
- It lets us analyse **stability** by looking at pole locations.
- It provides a **language** for combining systems (series, parallel, feedback) through transfer functions.
- The **convolution** operation in time becomes simple multiplication in $s$ (useful in signal processing).

This lecture expands on those ideas and gives you the tools to work fluently with the Laplace transform.

---

## Definition

The **Laplace transform** of a time-domain signal $f(t)$ is defined as

$$\mathcal{L}\{f\}(s) = \int_{0^-}^\infty f(t)\, e^{-st}\, dt$$

where $s = \sigma + j\omega$ is a **complex frequency** variable.

The lower limit $0^-$ ensures that impulses at $t=0$ are included. This matters for initial conditions.

### Why $s$?

The variable $s$ lives in the **complex plane** (the $s$-plane):
- The real part $\sigma = \text{Re}(s)$ governs **exponential growth or decay**.
- The imaginary part $\omega = \text{Im}(s)$ governs **oscillation frequency**.

A signal $e^{st} = e^{\sigma t} e^{j\omega t}$ grows/decays exponentially while oscillating.

This signal is the fundamental building block of linear system analysis.

---

## Laplace Transform Table

The table below lists the most common signals and their Laplace transforms.

In [13]:
from IPython.display import display, HTML, Math, Markdown
import sympy as sp

# ---- Build the table as an HTML string ----
# The rows: (time-domain expression, Laplace expression)
rows = [
    (r'\delta(t)', r'1'),
    (r'u(t)', r'\frac{1}{s}'),
    (r't\,u(t)', r'\frac{1}{s^{2}}'),
    (r't^{n}\,u(t)', r'\frac{n!}{s^{n+1}}'),
    (r'e^{-at}u(t)', r'\frac{1}{s+a}'),
    (r't e^{-at}u(t)', r'\frac{1}{(s+a)^{2}}'),
    (r'\sin(\omega t)u(t)', r'\frac{\omega}{s^{2}+\omega^{2}}'),
    (r'\cos(\omega t)u(t)', r'\frac{s}{s^{2}+\omega^{2}}'),
    (r'e^{-at}\sin(\omega t)u(t)', r'\frac{\omega}{(s+a)^{2}+\omega^{2}}'),
    (r'e^{-at}\cos(\omega t)u(t)', r'\frac{s+a}{(s+a)^{2}+\omega^{2}}'),
    (r'\frac{1}{b-a}(e^{-at}-e^{-bt})u(t)', r'\frac{1}{(s+a)(s+b)}'),
    (r'\frac{1}{\omega_n\sqrt{1-\zeta^2}}e^{-\zeta\omega_n t}\sin(\omega_n\sqrt{1-\zeta^2}\,t)u(t)',
     r'\frac{1}{s^{2}+2\zeta\omega_n s+\omega_n^{2}}'),
]

col_widths = ['52%', '48%']

html = '<table style="width:100%; border-collapse: collapse; font-size:14px;">'
html += '<thead><tr style="background-color:#f0f0f0; border-bottom:2px solid #333;">'
html += f'<th style="width:{col_widths[0]}; padding:10px; text-align:left;">Time Domain $f(t)$</th>'
html += f'<th style="width:{col_widths[1]}; padding:10px; text-align:left;">Laplace Domain $F(s)$</th>'
html += '</tr></thead><tbody>'

for i, (td, ld) in enumerate(rows):
    bg = ' style="background-color:#fafafa;"' if i % 2 == 0 else ''
    html += f'<tr{bg}>'
    html += f'<td style="padding:6px 10px; border-bottom:1px solid #ddd;">${td}$</td>'
    html += f'<td style="padding:6px 10px; border-bottom:1px solid #ddd;">${ld}$</td>'
    html += '</tr>'

html += '</tbody></table>'

display(HTML(html))

Time Domain $f(t)$,Laplace Domain $F(s)$
$\delta(t)$,$1$
$u(t)$,$\frac{1}{s}$
"$t\,u(t)$",$\frac{1}{s^{2}}$
"$t^{n}\,u(t)$",$\frac{n!}{s^{n+1}}$
$e^{-at}u(t)$,$\frac{1}{s+a}$
$t e^{-at}u(t)$,$\frac{1}{(s+a)^{2}}$
$\sin(\omega t)u(t)$,$\frac{\omega}{s^{2}+\omega^{2}}$
$\cos(\omega t)u(t)$,$\frac{s}{s^{2}+\omega^{2}}$
$e^{-at}\sin(\omega t)u(t)$,$\frac{\omega}{(s+a)^{2}+\omega^{2}}$
$e^{-at}\cos(\omega t)u(t)$,$\frac{s+a}{(s+a)^{2}+\omega^{2}}$


---

## Properties of the Laplace Transform

These properties let us manipulate signals and systems without going back to the definition integral every time.

### Properties Table

| Property | Time Domain | Laplace Domain | Notes |
|---|---|---|---|
| Linearity | $a f(t) + b g(t)$ | $a F(s) + b G(s)$ | Superposition holds |
| 1st-order derivative | $f'(t)$ | $s F(s) - f(0^-)$ | Initial condition appears |
| 2nd-order derivative | $f''(t)$ | $s^{2}F(s) - s f(0^-) - f'(0^-)$ | Two initial conditions |
| Integration | $\displaystyle\int_{0^-}^{t} f(\tau)\,d\tau$ | $\dfrac{1}{s} F(s)$ | Division by $s$ |
| Time shift | $f(t-T)u(t-T)$ | $e^{-sT}F(s)$ | Delay of $T$ seconds |
| Frequency shift | $e^{-at}f(t)$ | $F(s+a)$ | Exponential modulation |
| Time scaling | $f(at),\; a>0$ | $\dfrac{1}{a}F\!\left(\dfrac{s}{a}\right)$ | Compress/expand time |
| Differentiation in $s$ | $t f(t)$ | $-\dfrac{d}{ds}F(s)$ | Multiplication by $t$ |
| Initial value theorem | $\displaystyle\lim_{t\to 0^+} f(t)$ | $\displaystyle\lim_{s\to\infty} s F(s)$ |  |
| Final value theorem | $\displaystyle\lim_{t\to\infty} f(t)$ | $\displaystyle\lim_{s\to 0} s F(s)$ | steady-state value |

---

## Inverse Laplace Transform & Partial Fraction Expansion

To return from the $s$-domain to the time domain, we need the **inverse Laplace transform**:

$$f(t) = \mathcal{L}^{-1}\{F(s)\} = \frac{1}{2\pi j} \int_{\sigma-j\infty}^{\sigma+j\infty} F(s) e^{st} ds$$

In practice, we almost never evaluate this contour integral. Instead, we use **partial fraction expansion** to break $F(s)$ into terms that match entries in the Laplace table.

### Procedure

1. Write $F(s) = \frac{N(s)}{D(s)}$, ensuring the degree of $N$ is less than the degree of $D$ (if not, divide first).
2. Factor $D(s)$ into first-order and quadratic factors.
3. Expand into partial fractions.
4. Look up each term in the Laplace table.

### Interactive Partial Fraction Explorer

Enter a numerator and denominator in terms of $s$, and the tool will compute the partial fraction expansion and the corresponding time-domain signal.

In [ ]:
%pip install -q ipywidgets

In [14]:
import sympy as sp
import ipywidgets as widgets

# Pre-define symbols for reuse
s_sym2 = sp.symbols('s')
t_sym2 = sp.symbols('t', positive=True)

def pfe_and_inverse(num_str, den_str):
    """Given numerator and denominator strings, compute PFE and inverse."""
    try:
        N = sp.sympify(num_str)
        D = sp.sympify(den_str)
        F = N / D

        # Inverse Laplace
        f_t = sp.inverse_laplace_transform(F, s_sym2, t_sym2)

        # Partial fraction decomposition
        F_pfe = sp.apart(F, s_sym2)

        display(Math(r'F(s) = ' + sp.latex(F)))
        display(Math(r'{Partial fractions: } ' + sp.latex(F_pfe)))
        display(Math(r'f(t) = ' + sp.latex(f_t)))
    except Exception as e:
        print(f'Error: {e}')

num_input = widgets.Text(value='1', description='N(s):', continuous_update=False)
den_input = widgets.Text(value='s**2 + 3*s + 2', description='D(s):', continuous_update=False)
pfe_btn = widgets.Button(description='Compute', button_style='primary')
pfe_out = widgets.Output()

def on_pfe(b):
    with pfe_out:
        clear_output(wait=True)
        pfe_and_inverse(num_input.value, den_input.value)

pfe_btn.on_click(on_pfe)

display(widgets.VBox([
    widgets.HBox([num_input, den_input]),
    pfe_btn, pfe_out
]))
print("Try: N(s) = 1, D(s) = s**2 + 3*s + 2   →  f(t) = (e^{-t} - e^{-2t})u(t)")
print("Try: N(s) = s + 3, D(s) = s**2 + 2*s + 5  →  damped sinusoid")

Try: N(s) = 1, D(s) = s**2 + 3*s + 2   →  f(t) = (e^{-t} - e^{-2t})u(t)
Try: N(s) = s + 3, D(s) = s**2 + 2*s + 5  →  damped sinusoid


---

## Solving ODEs with the Laplace Transform

This is where the Laplace transform truly shines. Consider a second-order system:

$$\ddot y(t) + 3 \dot y(t) + 2 y(t) = u(t), \quad y(0) = 1,\; \dot y(0) = 0$$

**Step 1:** Transform both sides, incorporating initial conditions:

$$[s^2 Y(s) - s y(0) - \dot y(0)] + 3[s Y(s) - y(0)] + 2 Y(s) = \frac{1}{s}$$

**Step 2:** Substitute $y(0)=1,\; \dot y(0)=0$:

$$(s^2 + 3s + 2) Y(s) - (s + 3) = \frac{1}{s}$$

**Step 3:** Solve for $Y(s)$:

$$Y(s) = \frac{1}{s(s^2 + 3s + 2)} + \frac{s + 3}{s^2 + 3s + 2}$$

**Step 4:** Expand via partial fractions and invert.

---

## Initial Value Theorem (IVT) and Final Value Theorem (FVT)

These theorems let us extract $f(0^+)$ and $f(\infty)$ directly from $F(s)$, without computing the inverse transform.

### Initial Value Theorem

$$f(0^+) = \lim_{s \to \infty} s F(s)$$

Useful for checking that the initial conditions of your solution match.

### Final Value Theorem

If all poles of $s F(s)$ are in the open left half-plane (system is stable):

$$\lim_{t \to \infty} f(t) = \lim_{s \to 0} s F(s)$$

This is extremely useful in control: it tells us the **steady-state error** of a closed-loop system without solving for the full time response.

---

## The $s$-Plane and What Poles Tell Us

Every pole of $F(s)$ corresponds to a **mode** of the time-domain response:

| Pole location | Time-domain behavior | Example $F(s)$ | $f(t)$ |
|---|---|---|---|
| Real, $s = -\alpha$ | Exponential decay | $\frac{1}{s+\alpha}$ | $e^{-\alpha t}u(t)$ |
| Real, $s = 0$ | Step (constant) | $\frac{1}{s}$ | $u(t)$ |
| Real, $s = +\alpha$ | Exponential growth (unstable) | $\frac{1}{s-\alpha}$ | $e^{\alpha t}u(t)$ |
| Complex pair $s = -\alpha \pm j\omega$ | Damped oscillation | $\frac{\omega}{(s+\alpha)^2+\omega^2}$ | $e^{-\alpha t}\sin(\omega t)u(t)$ |
| Imaginary pair $s = \pm j\omega$ | Sustained oscillation | $\frac{\omega}{s^2+\omega^2}$ | $\sin(\omega t)u(t)$ |

The interactive $s$-plane explorer below lets you place poles and see the corresponding time response.

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

def pole_response_plot(sigma, omega):
    """Plot the s-plane and time response for poles at sigma ± j omega."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    # s-plane
    limit = max(abs(sigma) + 2, omega + 2, 3)
    ax1.axhline(0, color='k', linewidth=0.8)
    ax1.axvline(0, color='k', linewidth=0.8)
    ax1.set_xlim(-limit, limit)
    ax1.set_ylim(-limit, limit)
    ax1.set(xlabel='Re($s$)', ylabel='Im($s$)', title='$s$-Plane')
    ax1.grid(alpha=0.3)

    # Shade stability regions
    ax1.axvspan(0, limit, alpha=0.06, color='r')
    ax1.axvspan(-limit, 0, alpha=0.06, color='g')
    ax1.text(-limit*0.9, limit*0.85, 'Stable', fontsize=10, color='g', fontweight='bold')
    ax1.text(limit*0.05, limit*0.85, 'Unstable', fontsize=10, color='r', fontweight='bold')

    # Plot poles
    poles = [complex(sigma, omega), complex(sigma, -omega)]
    for p in poles:
        ax1.plot(p.real, p.imag, 'rx', markersize=12, markeredgewidth=3)
    ax1.axis('equal')

    # Time response (impulse response)
    t_vals = np.linspace(0, 5, 2000)
    if omega == 0:
        y_vals = np.exp(sigma * t_vals)
        label = f'$e^{{{sigma}t}}$'
    else:
        y_vals = np.exp(sigma * t_vals) * np.sin(omega * t_vals)
        label = f'$e^{{{sigma}t}} \\sin({omega}t)$'

    ax2.plot(t_vals, y_vals, 'b-', linewidth=1.5)
    ax2.axhline(0, color='k', alpha=0.2)
    ax2.set(xlabel='Time (s)', ylabel='$f(t)$', title=f'Impulse Response: {label}')
    ax2.grid(alpha=0.3)

    if sigma < 0:
        env = np.exp(sigma * t_vals)
        ax2.plot(t_vals, env, 'r--', alpha=0.4, linewidth=0.8)
        ax2.plot(t_vals, -env, 'r--', alpha=0.4, linewidth=0.8)

    fig.tight_layout()
    plt.show()

sigma_s = widgets.FloatSlider(min=-5, max=5, step=0.2, value=-0.5,
                              description='$\\sigma$ (real part):', style={'description_width': 'initial'})
omega_s2 = widgets.FloatSlider(min=0, max=10, step=0.5, value=4,
                               description='$\\omega$ (imag part):', style={'description_width': 'initial'})
run_pole_btn = widgets.Button(description='Plot', button_style='primary')
pole_out = widgets.Output()

def on_pole_plot(b):
    with pole_out:
        clear_output(wait=True)
        pole_response_plot(sigma_s.value, omega_s2.value)

run_pole_btn.on_click(on_pole_plot)
display(widgets.VBox([widgets.HBox([sigma_s, omega_s2]), run_pole_btn, pole_out]))
print("Try: σ = -0.5, ω = 4  →  damped oscillation (stable)")
print("Try: σ = 0, ω = 5     →  sustained oscillation")
print("Try: σ = 0.3, ω = 4   →  growing oscillation (unstable)")

Try: σ = -0.5, ω = 4  →  damped oscillation (stable)
Try: σ = 0, ω = 5     →  sustained oscillation
Try: σ = 0.3, ω = 4   →  growing oscillation (unstable)


---

## Summary

- The **Laplace transform** converts time-domain signals $f(t)$ to $s$-domain functions $F(s)$.
- The **ROC** (region of convergence) tells us where the integral is valid.
- The **table of transforms** and the **properties** are the practical tools for working in the $s$-domain.
- **Partial fraction expansion** is the primary method for computing inverse Laplace transforms.
- **ODEs** are solved by transforming, solving algebraically, and inverting.
- The **IVT** and **FVT** give quick access to initial and steady-state values.
- **Pole locations** in the $s$-plane directly determine the shape of the time response:
  - Left half-plane → stable (decaying)
  - Imaginary axis → sustained oscillation
  - Right half-plane → unstable (growing)